<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## Verificación y Validación de Modelos de Simulación
### T3.2 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Verificación vs. Validación — la distinción fundamental
2. **Ejemplo 1:** Verificación de una cola M/M/1 — trazas y conservación
3. Validación estadística: IC y prueba t
4. **Ejemplo 2:** Validación M/M/1 + detección de un bug (μ=4 vs μ=5)
5. Análisis de sensibilidad
6. **Ejemplo 3:** V&V completa de cola M/E₂/1 con sensibilidad

## Verificar vs. Validar
<div class='defn'>
<strong>Verificación:</strong> ¿Construimos el modelo <em>correctamente</em>? (ingeniería de software)<br>
<strong>Validación:</strong> ¿Construimos el modelo <em>correcto</em>? (modelado)
</div>

| | Verificación | Validación |
|---|---|---|
| **Pregunta** | ¿El código implementa el modelo? | ¿El modelo representa el sistema? |
| **Técnicas** | Trazas, conservación, aserciones | IC vs. fórmulas, test de Turing |
| **Detecta** | Bugs en código | Supuestos incorrectos |

<div class='alerta'>
Un modelo puede estar <strong>verificado</strong> (sin bugs) pero <strong>no validado</strong> (supuestos incorrectos), y viceversa.
</div>

## Ejemplo 1 — Verificación M/M/1 con SimPy
<div class='defn'>
Cola M/M/1 con λ=3 cl/min, μ=5 cl/min (ρ=0.6).
Objetivo: verificar que la implementación en SimPy es correcta.
</div>

**Pruebas de verificación:**
1. ✅ Traza de eventos (primeros 10 clientes)
2. ✅ Conservación de entidades
3. ✅ Prueba degenerada (λ=0)
4. ✅ Tráfico extremo (ρ→1)

In [ ]:
import simpy
import numpy as np
import matplotlib.pyplot as plt

def mm1_con_traza(lam, mu, T, seed=42, traza=False):
    """Simula M/M/1 y retorna métricas. Si traza=True, imprime eventos."""
    np.random.seed(seed)
    esperas, sistemas = [], []
    llegadas_tot, salidas_tot = 0, 0
    
    def cliente(env, srv, idx):
        nonlocal llegadas_tot, salidas_tot
        llegadas_tot += 1
        t_llegada = env.now
        with srv.request() as req:
            yield req
            espera = env.now - t_llegada
            esperas.append(espera)
            t_servicio = np.random.exponential(1/mu)
            if traza and idx <= 10:
                print(f'  C{idx:2d}: llega t={t_llegada:.3f}, espera={espera:.3f}, '
                      f'servicio={t_servicio:.3f}, sale t={env.now+t_servicio:.3f}, '
                      f'N={srv.count+len(srv.queue)}')
            yield env.timeout(t_servicio)
            sistemas.append(env.now - t_llegada)
            salidas_tot += 1
    
    def llegadas(env, srv):
        idx = 0
        while True:
            yield env.timeout(np.random.exponential(1/lam))
            idx += 1
            env.process(cliente(env, srv, idx))
    
    env = simpy.Environment()
    srv = simpy.Resource(env, capacity=1)
    env.process(llegadas(env, srv))
    env.run(until=T)
    
    return {'Wq': np.mean(esperas), 'W': np.mean(sistemas),
            'llegadas': llegadas_tot, 'salidas': salidas_tot,
            'en_sistema': llegadas_tot - salidas_tot,
            'esperas': esperas, 'sistemas': sistemas}

lam, mu = 3, 5
rho = lam/mu

# ─── Verificación 1: Traza de eventos ───
print('═══ TRAZA DE EVENTOS (primeros 10 clientes) ═══')
result_traza = mm1_con_traza(lam, mu, T=100, traza=True)

# ─── Verificación 2: Conservación ───
result = mm1_con_traza(lam, mu, T=2000)
print(f'\n═══ CONSERVACIÓN DE ENTIDADES (T=2000 min) ═══')
print(f'Llegadas: {result["llegadas"]}, Salidas: {result["salidas"]}, En sistema: {result["en_sistema"]}')
print(f'Llegadas = Salidas + En_sistema: {result["llegadas"]} = {result["salidas"]} + {result["en_sistema"]} ✓')

# ─── Verificación 3: Prueba degenerada (λ=0 simulado con λ muy baja) ───
result_deg = mm1_con_traza(0.001, mu, T=100)
print(f'\n═══ PRUEBA DEGENERADA (λ≈0) ═══')
print(f'Wq = {result_deg["Wq"]:.4f} (debe ser ≈0) ✓')

## Validación estadística
<div class='defn'>
Se ejecutan n réplicas → Y₁, ..., Yₙ (i.i.d.). Se compara con referencia θ₀:

$$IC_{95\%}(\theta) = \bar{Y}_n \pm t_{\alpha/2, n-1} \cdot \frac{S_n}{\sqrt{n}}$$

Si el IC contiene θ₀ → modelo consistente con la referencia.
</div>

**Fórmulas de referencia M/M/1:**
- W_q = ρ / [μ(1−ρ)]
- L_q = ρ² / (1−ρ)

<div class='nota'>
La ventaja del IC sobre la prueba t: muestra la <strong>magnitud</strong> de la discrepancia, no solo si es significativa.
</div>

## Ejemplo 2 — Validación M/M/1 y detección de un bug
<div class='defn'>
Mismo sistema M/M/1 (λ=3, μ=5). Se ejecutan 30 réplicas para validar.
Luego se muestra qué pasa cuando el código tiene un <strong>bug sutil</strong>: μ=4 en vez de μ=5.
</div>

**Pregunta clave:** ¿La verificación habría detectado este error?

*Spoiler: No. La verificación (trazas, conservación) pasa perfectamente con μ=4. Es la validación contra fórmulas la que lo detecta.*

In [ ]:
from scipy import stats as st

def replicas_mm1(lam, mu, T, n_rep):
    """Ejecuta n_rep réplicas y retorna vectores de Wq y rho por réplica."""
    Wq_vec, rho_vec = [], []
    for j in range(n_rep):
        r = mm1_con_traza(lam, mu, T, seed=1000+j)
        Wq_vec.append(r['Wq'])
        # Estimar rho como fraccion de tiempo ocupado
        rho_vec.append(sum(1 for w in r['sistemas'] if w > 0) * (1/mu) / T * lam)
    return np.array(Wq_vec), np.array(rho_vec)

lam, mu_correct, mu_bug = 3, 5, 4
rho_correct = lam/mu_correct
Wq_ref = rho_correct / (mu_correct * (1-rho_correct))

# ─── Modelo CORRECTO (μ=5) ───
Wq_ok, _ = replicas_mm1(lam, mu_correct, T=2000, n_rep=30)
mean_ok, se_ok = Wq_ok.mean(), Wq_ok.std(ddof=1)/np.sqrt(30)
t_crit = st.t.ppf(0.975, 29)
ci_ok = (mean_ok - t_crit*se_ok, mean_ok + t_crit*se_ok)

# ─── Modelo con BUG (μ=4) ───
Wq_bug, _ = replicas_mm1(lam, mu_bug, T=2000, n_rep=30)
mean_bug, se_bug = Wq_bug.mean(), Wq_bug.std(ddof=1)/np.sqrt(30)
ci_bug = (mean_bug - t_crit*se_bug, mean_bug + t_crit*se_bug)

print('═══ VALIDACIÓN: MODELO CORRECTO (μ=5) ═══')
print(f'Referencia analítica: Wq = {Wq_ref:.3f} min')
print(f'Simulado: Ȳ = {mean_ok:.3f} min,  IC 95%: [{ci_ok[0]:.3f}, {ci_ok[1]:.3f}]')
print(f'¿IC contiene {Wq_ref:.3f}? {"Sí ✓" if ci_ok[0] <= Wq_ref <= ci_ok[1] else "No ✗"}')
print(f'Error relativo: {abs(mean_ok-Wq_ref)/Wq_ref*100:.1f}%')

Wq_ref_bug = (lam/mu_bug) / (mu_bug * (1-lam/mu_bug))  # lo que realmente simula
print(f'\n═══ MODELO CON BUG (μ=4 en vez de 5) ═══')
print(f'Referencia (esperada si μ=5): Wq = {Wq_ref:.3f} min')
print(f'Simulado: Ȳ = {mean_bug:.3f} min,  IC 95%: [{ci_bug[0]:.3f}, {ci_bug[1]:.3f}]')
print(f'¿IC contiene {Wq_ref:.3f}? {"Sí" if ci_bug[0] <= Wq_ref <= ci_bug[1] else "No ✗ → BUG DETECTADO"}')
print(f'\n💡 ρ̂ ≈ {lam/mu_bug:.2f} = {lam}/{mu_bug} → apunta directamente a μ={mu_bug}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Modelo correcto
axes[0].hist(Wq_ok, bins=10, color='#1A7A9A', edgecolor='white', alpha=0.8)
axes[0].axvline(Wq_ref, color='#C62828', lw=2.5, ls='--', label=f'Analítico = {Wq_ref:.3f}')
axes[0].axvline(mean_ok, color='#1A7A9A', lw=2, label=f'Ȳ = {mean_ok:.3f}')
axes[0].axvspan(ci_ok[0], ci_ok[1], alpha=0.15, color='#1A7A9A', label='IC 95%')
axes[0].set_xlabel('Wq (min)'); axes[0].set_title('Modelo correcto (μ=5) — VALIDADO ✓')
axes[0].legend(fontsize=8)

# Modelo con bug
axes[1].hist(Wq_bug, bins=10, color='#C62828', edgecolor='white', alpha=0.8)
axes[1].axvline(Wq_ref, color='#2E7D32', lw=2.5, ls='--', label=f'Esperado = {Wq_ref:.3f}')
axes[1].axvline(mean_bug, color='#C62828', lw=2, label=f'Ȳ = {mean_bug:.3f}')
axes[1].axvspan(ci_bug[0], ci_bug[1], alpha=0.15, color='#C62828', label='IC 95%')
axes[1].set_xlabel('Wq (min)'); axes[1].set_title('Modelo con bug (μ=4) — BUG DETECTADO ✗')
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## Análisis de sensibilidad
<div class='defn'>
Verificar que el modelo responde en la <strong>dirección correcta</strong> y con <strong>magnitud razonable</strong> ante cambios en parámetros.
</div>

**Procedimiento:**
1. Variar cada parámetro ±10–20%
2. Verificar **dirección**: si λ↑ → Wq↑
3. Verificar **magnitud**: no-linealidad cerca de ρ→1
4. Si la dirección es incorrecta → **error en el modelo**

## Ejemplo 3 — V&V completa: Cola M/E₂/1
<div class='defn'>
Celda de manufactura: llegadas Poisson (λ=8 pzas/h), servicio Erlang-2 (E[S]=0.2 h, CV=0.71). ρ=0.8.
Proceso completo: verificación + validación P-K + análisis de sensibilidad.
</div>

**Referencia P-K:**
$$W_q = \frac{\lambda E[S^2]}{2(1-\rho)} = \frac{8 \times 0.06}{2 \times 0.2} = 1.20 \text{ h}$$

In [ ]:
def me2_1(lam, mu_fase, T, seed=42):
    """Simula M/E₂/1: servicio = suma de 2 exponenciales con tasa mu_fase."""
    np.random.seed(seed)
    esperas = []
    
    def cliente(env, srv):
        t_llegada = env.now
        with srv.request() as req:
            yield req
            esperas.append(env.now - t_llegada)
            # Servicio Erlang-2: dos fases exponenciales
            yield env.timeout(np.random.exponential(1/mu_fase))
            yield env.timeout(np.random.exponential(1/mu_fase))
    
    def llegadas(env, srv):
        while True:
            yield env.timeout(np.random.exponential(1/lam))
            env.process(cliente(env, srv))
    
    env = simpy.Environment()
    srv = simpy.Resource(env, capacity=1)
    env.process(llegadas(env, srv))
    env.run(until=T)
    return np.mean(esperas)

lam_m, mu_f = 8, 10  # pzas/h, fases/h
ES = 2/mu_f  # 0.2 h
ES2 = 2/mu_f**2 + ES**2  # 0.06 h²
rho_m = lam_m * ES  # 0.8
Wq_PK = lam_m * ES2 / (2*(1-rho_m))

# ─── Validación: 50 réplicas ───
n_rep = 50
Wq_reps = np.array([me2_1(lam_m, mu_f, T=2000, seed=2000+j) for j in range(n_rep)])
mean_m, se_m = Wq_reps.mean(), Wq_reps.std(ddof=1)/np.sqrt(n_rep)
t_c = st.t.ppf(0.975, n_rep-1)
ci_m = (mean_m - t_c*se_m, mean_m + t_c*se_m)

print('═══ VALIDACIÓN M/E₂/1 vs P-K ═══')
print(f'P-K analítico: Wq = {Wq_PK:.3f} h')
print(f'Simulado (n={n_rep}): Ȳ = {mean_m:.3f} h,  IC 95%: [{ci_m[0]:.3f}, {ci_m[1]:.3f}]')
print(f'¿IC contiene {Wq_PK:.3f}? {"Sí ✓" if ci_m[0] <= Wq_PK <= ci_m[1] else "No ✗"}')
print(f'Error relativo: {abs(mean_m-Wq_PK)/Wq_PK*100:.1f}%')

# ─── Análisis de sensibilidad: variar λ ±10%, ±20% ───
print(f'\n═══ ANÁLISIS DE SENSIBILIDAD ═══')
print(f'{"λ":>6} {"ρ":>6} {"Wq(P-K)":>10} {"Wq(sim)":>10} {"Dirección":>12}')
print('-'*48)
for factor in [0.8, 0.9, 1.0, 1.1, 1.2]:
    lam_v = lam_m * factor
    rho_v = lam_v * ES
    if rho_v >= 1: continue
    Wq_v = lam_v * ES2 / (2*(1-rho_v))
    Wq_sim_v = np.mean([me2_1(lam_v, mu_f, T=1000, seed=3000+j) for j in range(20)])
    direction = '—' if factor == 1.0 else ('↑ ✓' if Wq_sim_v > mean_m and factor > 1 else '↓ ✓' if factor < 1 else '?')
    label = '(base)' if factor == 1.0 else f'({factor-1:+.0%})'
    print(f'{lam_v:>6.1f} {rho_v:>6.2f} {Wq_v:>10.2f} {Wq_sim_v:>10.2f} {direction:>12} {label}')

In [ ]:
# Gráfico de sensibilidad: Wq vs ρ (analítico y simulado)
rho_range = np.linspace(0.5, 0.98, 50)
Wq_analitico = [(r/ES)*ES2/(2*(1-r)) for r in rho_range]

# Simulaciones puntuales
rho_pts = [0.64, 0.72, 0.80, 0.88, 0.96]
Wq_sim_pts = []
for rho_p in rho_pts:
    lam_p = rho_p / ES
    Wq_sim_pts.append(np.mean([me2_1(lam_p, mu_f, T=1000, seed=4000+j) for j in range(20)]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(rho_range, Wq_analitico, color='#1A7A9A', lw=2.5, label='P-K analítico')
axes[0].scatter(rho_pts, Wq_sim_pts, color='#C62828', s=80, zorder=5, label='Simulación')
axes[0].axvline(0.8, ls='--', color='gray', alpha=0.5, label='ρ base = 0.80')
axes[0].set_xlabel('ρ (utilización)'); axes[0].set_ylabel('Wq (h)')
axes[0].set_title('Sensibilidad: Wq vs ρ (M/E₂/1)'); axes[0].legend()
axes[0].set_ylim(0, 15)

# Histograma de las 50 réplicas con referencia
axes[1].hist(Wq_reps, bins=12, color='#1A7A9A', edgecolor='white', alpha=0.8)
axes[1].axvline(Wq_PK, color='#C62828', lw=2.5, ls='--', label=f'P-K = {Wq_PK:.2f} h')
axes[1].axvline(mean_m, color='#1A7A9A', lw=2, label=f'Ȳ = {mean_m:.2f} h')
axes[1].axvspan(ci_m[0], ci_m[1], alpha=0.15, color='#1A7A9A', label='IC 95%')
axes[1].set_xlabel('Wq (h)'); axes[1].set_title('Distribución de Wq (50 réplicas)')
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()
print('✓ Modelo verificado y validado. Listo para análisis avanzados.')

## Conclusiones

- **Verificación** (trazas, conservación, degeneradas) detecta bugs de lógica.
- **Validación** (IC vs. fórmulas) detecta errores de parametrización que la verificación no encuentra.
- El **análisis de sensibilidad** confirma que el modelo responde coherentemente y revela la no-linealidad cerca de ρ→1.
- Ambos procesos son **necesarios e insuficientes** por separado.
- La fórmula **P-K** (M/G/1) es la herramienta ideal de validación: verifica el promedio, y si coincide, confiamos en percentiles y otras métricas simuladas.

**Próximo tema:** T3.3 — Análisis de salida: simulaciones terminales.